# Ghost Hunting with Bayesian Inference in TextWorld

An agent explores a grid-shaped text adventure game and locates a hidden ghost using only two noisy distance sensors. It keeps a probability for every grid cell and updates it with Bayes' rule after each move, reporting the ghost's position once one cell reaches the confidence threshold `alpha`.

See the [README](README.md) for a step-by-step explanation.

## Installation

TextWorld-Express needs Java 1.8 or newer.

In [ ]:
%pip install gymnasium textworld-express numpy

## Imports

In [13]:
# export - DO NOT MODIFY THIS CEL͏󠄀͏󠄍͏︃͏︆͏︀͏︀͏︋͏󠄍͏󠄇͏󠄀͏󠄎͏󠄒͏︎͏󠄇͏︈͏︃͏︁L
import gymnasium
import numpy as np
from helpers import GhostTextWorldExpressEnv

## Ghost Hunting Overview

This environment features a ghost that cannot be seen and might be inside a room or inside the walls (not in a room). However, you can "hear" the ghost make noise. Whenever an action is taken in the world, the agent will also receive information about how far away the ghost sounds. Unfortunately, this perceived distance is noisy and sometimes it will sound farther away than it actually is, and sometimes it will sound closer than it actually is.

**Luckily, we also have another tool in our ghost hunting toolbelt**: We are given a special sensor that can give us a distribution of the horizonal (X coordinate) distance of the ghost from the origin of our grid.

The agent's goal is to locate where the ghost is.

The `infos` returned by the `env.step()` action returns many useful pieces of information, including the following:
- `infos['ghost']` holds a distance (Manhattan distance) to the ghost. This is a "noisy" distance--it may not be the actual distance to the ghost.
- `infos['distribution']` holds a distribution over possible noisy distance values, as a numpy array where the index is the number of units away (Manhattan distance). For example, if `infos['ghost'] = 6` then the distribution will tell you the probability that that value is accurate. If the distribution is:
```
[0.         0.0106383  0.0212766  0.04255319 0.08510638 0.17021277 0.34042553 0.17021277 0.08510638 0.04255319 0.0212766 ]
```
then there is ~34% probability that the ghost is actually 6 units away, but there is ~17% probability that the ghost is actually 5 units away.
- `infos['x_dist_distribution']` holds the distribution of the noisy x distance of the ghost from the origin, as a numpy array where the index is the number of units away (Manhattan distance). This operates the same way as the distribution explained above, but for x distance rather than manhattan distance.
- `infos['player']` holds the `(x, y)` position of the player/agent.
- `infos['validActions']` holds the list of valid actions that can be taken from the current state.

Feel free to print out `infos` to see what all the information looks like. 

The agent must guess the `(x, y)` position of the ghost. Once it has a reliable guess, it can perform a special action, `'report'`, which is followed by two numbers separated by spaces that indicate the ghost's x and y position (e.g., `'report 3 4'` would indicate that the agent thought the ghost was in position x=3, y=4). If the guess is correct, the observation returned will be `'True'` and if it is wrong, the observation returned will be `'False'`. **Once a report is made, the agent cannot perform any further actions.**

## Environment Setup

New environments must be registered through the Gymnasium API.

In [15]:
gymnasium.register(id='TextWorldExpress-GhostTextWorldExpressEnv-v0', entry_point='__main__:GhostTextWorldExpressEnv')

Set the game type and game parameters, and load the environment information into global variables (described below). 

**Note:** We have provided samples for the `GAME_PARAMS` variables. While we do provide a function to test your search algorithms with each of these environments automatically, to closely and thoroughly test and verify the correctness of your implementations, we recommend loading other environemnts and testing your code with then. 

In [16]:
# set the seed and create the environmen͏󠄀͏󠄍͏︃͏︆͏︀͏︀͏︋͏󠄍͏󠄇͏󠄀͏󠄎͏󠄒͏︎͏󠄇͏︈͏︃͏︁t
SEED = 3
ENV = GhostTextWorldExpressEnv(envStepLimit=100)

# set the game type and parameter͏󠄀͏󠄍͏︃͏︆͏︀͏︀͏︋͏󠄍͏󠄇͏󠄀͏󠄎͏󠄒͏︎͏󠄇͏︈͏︃͏︁s
# change the game type and parameters here to test different environment͏󠄀͏󠄍͏︃͏︆͏︀͏︀͏︋͏󠄍͏󠄇͏󠄀͏󠄎͏󠄒͏︎͏󠄇͏︈͏︃͏︁s
GAME_TYPE = "coin"
GAME_PARAMS = "numLocations=5,includeDoors=1,numDistractorItems=0"
# GAME_TYPE="mapreader͏󠄀͏󠄍͏︃͏︆͏︀͏︀͏︋͏󠄍͏󠄇͏󠄀͏󠄎͏󠄒͏︎͏󠄇͏︈͏︃͏︁"
# GAME_PARAMS="numLocations=5,maxDistanceApart=3,includeDoors=0,maxDistractorItemsPerLocation=0͏󠄀͏󠄍͏︃͏︆͏︀͏︀͏︋͏󠄍͏󠄇͏󠄀͏󠄎͏󠄒͏︎͏󠄇͏︈͏︃͏︁"

# load the evironmen͏󠄀͏󠄍͏︃͏︆͏︀͏︀͏︋͏󠄍͏󠄇͏󠄀͏󠄎͏󠄒͏︎͏󠄇͏︈͏︃͏︁t
ENV.load(gameName=GAME_TYPE, gameParams=GAME_PARAMS)

INFO:py4j.java_gateway:Callback Server Starting
INFO:py4j.java_gateway:Socket listening on ('127.0.0.1', 36303)


Reset the environment (same as before).

In [17]:
obs, infos = ENV.reset(seed=SEED, gameFold="train", generateGoldPath=True)
print(obs)
print(infos["validActions"])
print('player:', infos['player'])
print('ghost noisy distance:', infos['ghost'])
print('distribution:', infos['distribution'])
print('distribution:', infos['x_dist_distribution'])

self.__ghost:  (-5, -4)
You are in the kitchen. In one part of the room you see a stove. There is also an oven. You also see a fridge that is closed. In another part of the room you see a counter, that has nothing on it. In one part of the room you see a kitchen cupboard that is closed. There is also a cutlery drawer that is closed. You also see a trash can that is closed. In another part of the room you see a dishwasher that is closed. In one part of the room you see a dining chair, that has nothing on it. 
To the South you see a closed sliding patio door. To the West you see a closed frosted-glass door. 
You hear a ghost 9.00 rooms away.
['look around', 'close door to west', 'move west', 'open door to south', 'open door to west', 'inventory', 'move south', 'close door to south']
player: (0, 0)
ghost noisy distance: 9
distribution: [0.         0.         0.         0.         0.0106383  0.0212766
 0.04255319 0.08510638 0.17021277 0.34042553 0.17021277 0.08510638
 0.04255319 0.0212766 

## The Agent

`run_agent` explores the map, updates a belief over every grid cell after each move, and returns the most likely ghost position once its probability exceeds `alpha`.

In [22]:
# export - DO NOT MODIFY OR MOVE THIS LIN͏󠄀͏󠄍͏︃͏︆͏︀͏︀͏︋͏󠄍͏󠄇͏󠄀͏󠄎͏󠄒͏︎͏󠄇͏︈͏︃͏︁E
def run_agent(
    env: GhostTextWorldExpressEnv, obs: str, infos: dict, alpha: float = 0.5
) -> tuple[int, int]:
    guess_x, guess_y = None, None
    ### YOUR CODE BELOW HERE ##͏󠄀͏󠄍͏︃͏︆͏︀͏︀͏︋͏󠄍͏󠄇͏󠄀͏󠄎͏󠄒͏︎͏󠄇͏︈͏︃͏︁#
    
    dim = 2 * env.get_max_dist() + 1
        
    coords = [(i, j) for i in range(dim) for j in range(dim)]
        
    curr_infos = open_all_door(env, infos)

    dist = np.full(len(coords), 1 / (dim ** 2))
    # the distribution of (X:= state that the ghost will be in) and observations
    beta = np.full(len(coords), -1)

    
    t = 0
    
    results = []
    while t < 150000:
        # make a random move
        curr_infos = open_all_door(env, curr_infos)
        move_list = [action for action in curr_infos["validActions"] if "move" in action]
        curr_infos = env.step(np.random.choice(move_list))[3]
        
        # make observed matrix
        player_coord = (curr_infos["player"][0] + env.get_max_dist(), curr_infos["player"][1] + env.get_max_dist())
        obs_M_dist = curr_infos["distribution"]
        obs_X_dist = curr_infos["x_dist_distribution"]
        
        obs_M = np.array([obs_M_dist[cal_manhattan_dist(player_coord, coords[i])] for i in range(len(coords))])
        obs_X = np.array([obs_X_dist[abs(coords[i][0] - env.get_max_dist())] for i in range(len(coords))])

        # calculate beta for obs_X_dist
        ## beta zero:

        if t == 0:
            beta = obs_X * dist
        else:
            beta = obs_X * beta
        
        ## distribution of P(X_t | Y_0:t)

        if np.sum(beta) != 0:
            dist = beta / np.sum(beta)
        if np.any(dist > alpha):
            results = np.where(dist > alpha)[0]
        
        ## convergence
        if len(results) == 1:
            break
        
        # calculate beta for obs_M_dist
        ## beta zero:
        
        if t == 0:
            beta = obs_M * dist
        else:
            beta = obs_M * beta
                    
        ## distribution of P(X_t | Y_0:t)
        
        if np.sum(beta) != 0:
            dist = beta / np.sum(beta)
        if np.any(dist > alpha):
            results = np.where(dist > alpha)[0]
            
        ## convergence
        if len(results) == 1:
            break
        
        t += 1
    
    if len(results) == 0:
        random_id = np.random.choice(len(coords))
        x, y = coords[random_id]
        guess_x, guess_y = x - env.get_max_dist(), y - env.get_max_dist()
    else:
        guess_x, guess_y = coords[results[0]][0] - env.get_max_dist(), coords[results[0]][1] - env.get_max_dist() 

    ### YOUR CODE ABOVE HERE ##͏󠄀͏󠄍͏︃͏︆͏︀͏︀͏︋͏󠄍͏󠄇͏󠄀͏󠄎͏󠄒͏︎͏󠄇͏︈͏︃͏︁#
    return guess_x, guess_y

def cal_manhattan_dist(player_coord: tuple[int, int], curr_coord: tuple[int, int]):
    return abs(player_coord[0] - curr_coord[0]) + abs(player_coord[1] - curr_coord[1])
    
    
def open_all_door(env: GhostTextWorldExpressEnv, curr_infos: dict):
    open_list = [action for action in curr_infos["validActions"] if "open" in action]
    
    for action in open_list:
        curr_infos = env.step(action)[3]
    
    return curr_infos

## Testing Your Agent

In [23]:
# change the game type and parameters here to test different environment͏󠄀͏󠄍͏︃͏︆͏︀͏︀͏︋͏󠄍͏󠄇͏󠄀͏󠄎͏󠄒͏︎͏󠄇͏︈͏︃͏︁s
GAME_TYPE = "coin"
GAME_PARAMS = "numLocations=5,includeDoors=1,numDistractorItems=0"
# GAME_TYPE="mapreader͏󠄀͏󠄍͏︃͏︆͏︀͏︀͏︋͏󠄍͏󠄇͏󠄀͏󠄎͏󠄒͏︎͏󠄇͏︈͏︃͏︁"
# GAME_PARAMS="numLocations=5,maxDistanceApart=3,includeDoors=0,maxDistractorItemsPerLocation=0͏󠄀͏󠄍͏︃͏︆͏︀͏︀͏︋͏󠄍͏󠄇͏󠄀͏󠄎͏󠄒͏︎͏󠄇͏︈͏︃͏︁"

ENV.load(gameName=GAME_TYPE, gameParams=GAME_PARAMS)

# Run agent and see if it locates the ghos͏󠄀͏󠄍͏︃͏︆͏︀͏︀͏︋͏󠄍͏󠄇͏󠄀͏󠄎͏󠄒͏︎͏󠄇͏︈͏︃͏︁t
obs, infos = ENV.reset(0)
x, y = run_agent(ENV, obs, infos, alpha = 0.9)
print(x, y)

self.__ghost:  (-1, -2)
-1 -2


Testing the agent on all provided configurations.

In [24]:
from tests import test_all_configs

test_all_configs(run_agent=run_agent, alpha_param=0.95)

coin numLocations=5,includeDoors=1,numDistractorItems=0 0
self.__ghost:  (-3, 3)
Guess: ( -3 ,  3 )
coin numLocations=5,includeDoors=1,numDistractorItems=0 1
self.__ghost:  (0, -1)
Guess: ( 0 ,  -1 )
coin numLocations=5,includeDoors=1,numDistractorItems=0 2
self.__ghost:  (-2, 0)
Guess: ( -2 ,  0 )
coin numLocations=5,includeDoors=1,numDistractorItems=0 3
self.__ghost:  (1, 4)
Guess: ( 1 ,  4 )
coin numLocations=5,includeDoors=1,numDistractorItems=0 4
self.__ghost:  (-3, 3)
Guess: ( -3 ,  3 )
coin numLocations=5,includeDoors=1,numDistractorItems=0 5
self.__ghost:  (3, 4)
Guess: ( 3 ,  4 )
coin numLocations=5,includeDoors=1,numDistractorItems=0 6
self.__ghost:  (0, 1)
Guess: ( 0 ,  1 )
coin numLocations=5,includeDoors=1,numDistractorItems=0 7
self.__ghost:  (-3, 1)
Guess: ( -3 ,  1 )
coin numLocations=5,includeDoors=1,numDistractorItems=0 8
self.__ghost:  (-5, -5)
Guess: ( -5 ,  -5 )
coin numLocations=5,includeDoors=1,numDistractorItems=0 9
self.__ghost:  (-2, 5)
Guess: ( -2 ,  5 )
coin